In [ ]:
from ckonlpy.tag import Twitter
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

#### 전처리 대상 데이터 가져오기

##### 민원사무처리기준표 데이터를 기준으로 서류사전 구축

In [ ]:
raw_data = pd.read_excel('../P1_Data/02.Pretreatment/민원사무처리기준표_predata.xlsx',header = 0) #엑셀데이터 형태상 header지정
data = raw_data.copy()
data

In [ ]:
use_col = data.loc[:, ['구비서류']]
use_col = use_col[use_col['구비서류'].notnull()] #특정컬럼 널값제거
use_col

#### 서류항목 전처리 

##### 줄바꿈을 기준으로 row 늘리기

In [ ]:
# def split_rows(df, col_name):
#     new_df = df.copy()
#     new_df[col_name] = new_df[col_name].apply(lambda x: x.split('\n'))
#     new_df = new_df.explode(col_name)
#     new_df = new_df[new_df[col_name].notnull()]
#     return new_df

In [ ]:
# split_data = split_rows(use_col,'구비서류')
# split_data

#### 1.(1) 등과 같이 괄호 숫자로 시작하는 row 추출

In [ ]:
pattern = r'\([0-9]+\)\s*.*'
split_data_pre = use_col['구비서류'].apply(lambda x: re.findall(pattern, str(x))) # 정규표현식 패턴을 이용하여 괄호 안의 숫자와 이어지는 모든 문자열을 추출
split_data_pre = split_data_pre.apply(lambda x: x[0] if len(x) > 0 else None)
split_data_pre = use_col[split_data_pre.notnull()] #추출되지 않은 경우 None으로 대체
split_data_pre

#### 2. (),{} 등 괄호 및 괄호 내용 삭제 함수

    주어진 데이터프레임에서 주어진 열(column_name)에 대해 특수 문자나 괄호 등을 삭제하여 전처리

    Parameters:
        - df: 전처리할 데이터프레임
        - column_name: 전처리할 열 이름

    Returns:
        - 전처리된 데이터프레임

In [ ]:
def preprocess_data(df, column_name):
    # 괄호와 괄호 안의 내용을 삭제하는 정규표현식 패턴
    pattern = r'\([^)]*\)|\{[^}]*\}|\「[^」]*\」|\｢[^」]*\」|\｢[^｣]*\｣|\b[가-하][\)\]][^)]*|\b\d+\)'
    # document 열에서 괄호와 괄호 안의 내용을 삭제
    df[column_name +'_전처리'] = df[column_name].apply(lambda x: re.sub(pattern, '', x))
    return df

In [ ]:
split_data_pre = preprocess_data(split_data_pre,'구비서류') #함수호출
split_data_pre


    주어진 데이터프레임에서 주어진 열(col_name)의 각 행을 구분자(sep_list)로 분리하여 새로운 열(new_col_name)을 만듭니다.

    Parameters:
        - df: 분리 작업을 수행할 데이터프레임
        - col_name: 분리 작업을 수행할 열 이름
        - sep_list: 열의 각 행을 분리할 구분자 리스트
        - new_col_name: 분리 작업 결과를 저장할 새로운 열 이름

    Returns:
        - 분리 작업 결과를 저장한 새로운 데이터프레임


In [ ]:
def split_rows_list(df, col_name, sep_list, new_col_name):
    df = df[df[col_name].notnull()] # 결측치가 아닌 행만 선택
    new_df = df.copy() # 기존 데이터프레임을 복사하여 새로운 데이터프레임을 생성
    sep_regex = '|'.join(map(re.escape, sep_list)) # 구분자 리스트를 이용하여 정규표현식 패턴 생성
    new_df[new_col_name] = new_df[col_name].str.replace(sep_regex, ',').str.split(',') # 구분자를 쉼표로 대체하여 새로운 열 생성
    new_df = new_df.explode(new_col_name) # 새로운 열을 행 단위로 나누어서 데이터프레임 생성
    return new_df

In [ ]:
list = ['및', '또는', ',','사본'] # 구분자 리스트 생성
split_data_pre2 = split_rows_list(split_data_pre, '구비서류_전처리', list, '구비서류_전처리_split') #함수호출
split_data_pre2

#### 3어절 이하 추출

In [ ]:
split_data_pre_split = split_data_pre2[['구비서류_전처리_split']]
split_data_pre_3down = split_data_pre_split[split_data_pre_split['구비서류_전처리_split'].str.split().str.len() <= 3] # 3어절 이하 추출
split_data_pre_3down

#### 트위터 패키지 고유명사 등록 후 불용어 사전 적용

In [ ]:
file_path = '../P1_Data/01.External/text_dictionary/userword.txt'  # 메모장 파일 경로

with open(file_path, 'r', encoding='utf-8') as file:     # 사용되는 명사가 담긴 메모장 파일 열기
    contents = file.readlines()       # 파일 내용 읽어오기

# 메모장 파일에서 읽어온 내용을 리스트에 추가
text_list = []
for line in contents:
    text_list.append(line.strip())

## 사전에 등록한 단어를 고유명사 등록
twitter = Twitter()                            # 트위터 형태소 분석기 객체 생성 후, 리스트에 등록한 단어를 고유명사로 등록
twitter.add_dictionary(text_list, 'Noun')

# 불용어 사전 불러오기
with open('../P1_Data/01.External/text_dictionary/stopword.txt', 'r', encoding='utf-8') as f: #불용어 사전
    stopwords = f.read().split()

# 전처리된 데이터프레임에 등록한 단어를 토대로 고유명사 분석 후, 불용어를 제외한 단어로 구성된 새로운 열 추가
split_data_pre_3down['구비서류_전처리_okt적용'] = split_data_pre_3down['구비서류_전처리_split'].apply(lambda x: ' '.join([word for word in twitter.morphs(x) if word not in stopwords]))
split_data_pre_3down

In [ ]:
# 지정한 컬럼에서 정규식 표현의 한글만 추출
split_data_pre_3down['구비서류_전처리_okt적용_한글추출'] = split_data_pre_3down['구비서류_전처리_okt적용'].apply(lambda x: re.sub('[^가-힣]', '', str(x)))
split_data_pre_3down

In [ ]:
split_data_3down_korean = split_data_pre_3down[['구비서류_전처리_okt적용_한글추출']] #전처리된 한글 텍스트 추출하는 열만 남기고 나머지 열 제거
split_data_3down_korean = split_data_3down_korean.replace('', np.nan).dropna() #빈칸은 NaN으로 대체한 뒤, NaN을 포함하는 행 제거
split_data_3down_korean.replace(to_replace=r'(조제항|조제항제호|법조제항제호|제호|법조제항|조제항제|영제조제항|조제항제호|제조제항|호에|해당하는|하였음)', value = '', regex = True, inplace = True) # 정규표현식을 사용하여 특정 문자열 제거 (해당하는, 조제항, 조제항제, 조제항제호, 조제항제호에, 법조제항, 법조제항제호, 제조제항, 제호, 영제조제항, 하였음)

In [ ]:
split_data_oktpre_dupli = split_data_3down_korean.drop_duplicates(subset=['구비서류_전처리_okt적용_한글추출']) #중복제거
split_data_oktpre_dupli

In [ ]:
split_data_oktpre_3up = split_data_oktpre_dupli[split_data_oktpre_dupli['구비서류_전처리_okt적용_한글추출'].apply(lambda x: len(x) > 4)]
split_data_oktpre_3up #열의 길이가 4 이상인 행들을 추출하여 'split_data_oktpre_3up' 데이터프레임에 저장

In [ ]:
split_data_oktpre_sorted = split_data_oktpre_3up.sort_values(by='구비서류_전처리_okt적용_한글추출', ascending=True)
split_data_oktpre_sorted.reset_index(drop = True, inplace = True)
split_data_oktpre_sorted

In [ ]:
split_data_oktpre_sorted.to_excel('../P1_Data/02.Pretreatment/01_01_result_dic_stopword.xlsx')

#### 1차 사전 생성 (dictionary_9)

In [ ]:
doc_list = pd.read_excel('../P1_Data/02.Pretreatment/01_01_result_dic_stopword.xlsx',header = 0, index_col=0) #구비서류 목록 사전(민원서류기준표)

dictionary_9 = doc_list.copy()
dictionary_9 = dictionary_9.drop_duplicates(['구비서류_전처리_okt적용_한글추출']) #중복제거
dictionary_9

In [ ]:
file_path = '../P1_Data/01.External/text_dictionary/delete_dictionary.txt'  # 메모장 파일 경로

with open(file_path, 'r', encoding='utf-8') as file:  # 메모장 파일 열기
    contents = file.readlines()  # 파일 내용 읽어오기

# 리스트에 파일 내용 추가
aaa = []
for line in contents:
    aaa.append(line.strip())

for item in aaa:
    dictionary_9 = dictionary_9.drop(dictionary_9[dictionary_9['구비서류_전처리_okt적용_한글추출'] == item].index)
    #리스트 안에 있는 각각의 단어들에 대해 dictionary_9에서 해당 단어를 가진 행을 삭제

In [ ]:
dictionary_9

In [ ]:
dictionary_9.to_excel('../P1_Data/02.Pretreatment/01_01_dictionary_9.xlsx')

In [ ]:
dictionary_9